In [1]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# 1. File names

input_file="Exoplanet_6059_with_SIMBAD_fill_Finale.xlsx"
output_file="Exoplanet_6059_with_derived_spectral_columns_Finale.xlsx"

# 2. Load data

df=pd.read_excel(input_file)

# Cleanup
df=df.rename(columns={"StellarRadius [Solar Radius]": "Stellar Radius [Solar Radius]"})

# 3. Helper functions

def spectral_letter(teff):
    if pd.isna(teff):
        return None
    if teff >= 30000:
        return "O"
    elif teff >= 10000:
        return "B"
    elif teff >= 7500:
        return "A"
    elif teff >= 6000:
        return "F"
    elif teff >= 5200:
        return "G"
    elif teff >= 3700:
        return "K"
    else:
        return "M"

def spectral_subclass(teff):
    if pd.isna(teff):
        return None

    # Approximate subclass mapping from hot to cool within each class
    bands = [
        ("O", 30000, 50000),
        ("B", 10000, 30000),
        ("A", 7500, 10000),
        ("F", 6000, 7500),
        ("G", 5200, 6000),
        ("K", 3700, 5200),
        ("M", 2400, 3700),
    ]

    for letter, low, high in bands:
        if low <= teff < high or (letter == "O" and teff >= low):
            # hotter stars get smaller subclass numbers
            top=high
            bottom=low

            # avoid divide-by-zero
            if top==bottom:
                return f"{letter}5"

            frac=(top - teff)/(top - bottom)

            # clamp to 0..0.999...
            frac=max(0, min(frac, 0.999999))
            subclass=int(frac * 10)

            # keep subclass 0-9
            subclass=min(max(subclass, 0), 9)
            return f"{letter}{subclass}"

    # Cooler than M-band lower bound
    if teff<2400:
        return "M9"

    return None

def estimate_luminosity_class(logg, radius):
    if pd.isna(logg) and pd.isna(radius):
        return None

    # Very rough rule-based estimate
    if pd.notna(logg):
        if logg >= 4.0:
            return "V"
        elif 3.0 <= logg < 4.0:
            return "IV"
        elif 1.5 <= logg < 3.0:
            return "III"
        elif logg < 1.5:
            return "I"

    # Fallback if logg missing, use radius
    if pd.notna(radius):
        if radius <= 1.8:
            return "V"
        elif radius <= 4:
            return "IV"
        elif radius <= 20:
            return "III"
        else:
            return "I"

    return None

def clean_observed_sptype(val):
    if pd.isna(val):
        return None
    s=str(val).strip()
    if s == "" or s.lower() == "nan":
        return None
    return s

# 4. Build derived columns
derived_subclass=[]
estimated_lumclass=[]
derived_full=[]
final_sptype=[]
source_label=[]

for _, row in df.iterrows():
    observed = clean_observed_sptype(row.get("Star Spectral Type"))
    teff = row.get("Stellar Effective Temperature [K]")
    logg = row.get("Stellar Surface Gravity [log10(cm/s**2)]")
    radius = row.get("Stellar Radius [Solar Radius]")

    subclass = spectral_subclass(teff)
    lumclass = estimate_luminosity_class(logg, radius)

    if subclass is not None and lumclass is not None:
        full = f"{subclass} {lumclass}"
    elif subclass is not None:
        full = subclass
    else:
        full = None

    if observed is not None:
        final_type = observed
        source = "Observed"
    elif full is not None:
        final_type = full
        source = "Derived"
    else:
        final_type = None
        source = None

    derived_subclass.append(subclass)
    estimated_lumclass.append(lumclass)
    derived_full.append(full)
    final_sptype.append(final_type)
    source_label.append(source)

df["Derived Spectral Subclass"]=derived_subclass
df["Estimated Luminosity Class"]=estimated_lumclass
df["Derived Spectral Type"]=derived_full
df["Final Spectral Type"]=final_sptype
df["Final Spectral Type Source"]=source_label

# 5. Summary counts

observed_count=(df["Final Spectral Type Source"] == "Observed").sum()
derived_count=(df["Final Spectral Type Source"] == "Derived").sum()
missing_final_count=df["Final Spectral Type"].isna().sum()

# 6. Save workbook

df.to_excel(output_file, index=False)

# 7. Highlight derived final types

wb=load_workbook(output_file)
ws=wb.active

orange_fill=PatternFill(fill_type="solid", fgColor="FCE4D6")

header_map={cell.value: cell.column for cell in ws[1]}

for row_idx in range(2, len(df) + 2):
    idx = row_idx - 2
    if df.iloc[idx]["Final Spectral Type Source"] == "Derived":
        col_idx = header_map["Final Spectral Type"]
        ws.cell(row=row_idx, column=col_idx).fill = orange_fill

# 8. Add summary sheet

summary_sheet_name="Spectral Type Summary"
if summary_sheet_name in wb.sheetnames:
    del wb[summary_sheet_name]

summary=wb.create_sheet(summary_sheet_name)

summary["A1"]="Output File"
summary["B1"]=output_file

summary["A3"]="Count Type"
summary["B3"]="Value"

summary["A4"]="Observed Final Spectral Types"
summary["B4"]=int(observed_count)

summary["A5"]="Derived Final Spectral Types"
summary["B5"]=int(derived_count)

summary["A6"]="Missing Final Spectral Types"
summary["B6"]=int(missing_final_count)

summary["A8"]="Derived Columns"
summary["B8"]="Derived Spectral Subclass, Estimated Luminosity Class, Derived Spectral Type, Final Spectral Type, Final Spectral Type Source"

summary["A10"]="Color Key"
summary["B10"]="Orange = Final Spectral Type derived from temperature/logg/radius"

# 9. Save workbook

wb.save(output_file)

# 10. Print results

print("Done.")
print("Saved:", output_file)
print("Observed final spectral types:", observed_count)
print("Derived final spectral types:", derived_count)
print("Missing final spectral types:", missing_final_count)

Done.
Saved: Exoplanet_6059_with_derived_spectral_columns_Finale.xlsx
Observed final spectral types: 3856
Derived final spectral types: 1960
Missing final spectral types: 243
